# MobileViT with Hugging Face Transformers

This notebook is a hands-on tutorial for the `transformers` implementation of MobileViT, with a side-by-side runtime comparison against a standard ViT checkpoint.

Official references used while building this notebook:
- [MobileViT docs](https://huggingface.co/docs/transformers/model_doc/mobilevit)
- [ViT docs](https://huggingface.co/docs/transformers/model_doc/vit)
- [MobileViT source](https://github.com/huggingface/transformers/blob/main/src/transformers/models/mobilevit/modeling_mobilevit.py)
- [ViT source](https://github.com/huggingface/transformers/blob/main/src/transformers/models/vit/modeling_vit.py)

What we cover:
1. Important class and method definitions.
2. Loading a pretrained checkpoint.
3. A tensor walkthrough from image to logits with dimensions printed at each step.
4. A runtime comparison of MobileViT vs. standard ViT on the same image.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# If you are running in a fresh notebook, install the core libraries once:
# %pip install -q transformers pillow matplotlib

import inspect
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from transformers import AutoImageProcessor, MobileViTForImageClassification, ViTForImageClassification
from transformers.models.mobilevit.modeling_mobilevit import MobileViTLayer, MobileViTModel
from transformers.models.vit.modeling_vit import ViTEmbeddings, ViTLayer, ViTModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

def make_demo_image(size: int = 224) -> Image.Image:
    axis = np.linspace(0, 255, size, dtype=np.uint8)
    grid_x, grid_y = np.meshgrid(axis, axis)
    rgb = np.stack([grid_x, grid_y, 255 - grid_x], axis=-1)
    return Image.fromarray(rgb, mode='RGB')

demo_image = make_demo_image(224)

plt.figure(figsize=(4, 4))
plt.imshow(demo_image)
plt.axis('off')
plt.title('Synthetic demo image')
plt.show()

def shape_of(value):
    if isinstance(value, torch.Tensor):
        return tuple(value.shape)
    if isinstance(value, (list, tuple)):
        return [shape_of(item) for item in value]
    return type(value).__name__

def print_shape(label, value):
    print(f'{label:<50} {shape_of(value)}')


## Important classes and methods

For MobileViT, the key pieces are the convolutional stem, the MobileNet-style stages, the MobileViT block that unfolds feature maps into patches, and the final pooling/classification head.

For ViT, the important parts are the patch embeddings, the encoder layer, and the image-classification head.


In [ ]:
def show_signature(obj):
    print(f'{obj.__qualname__}{inspect.signature(obj)}')

def show_source_snippet(obj, lines: int = 24):
    print(f'\n### {obj.__qualname__}{inspect.signature(obj)}')
    snippet = inspect.getsource(obj).splitlines()[:lines]
    print('\n'.join(snippet))

print('Class signatures')
for cls in [
    MobileViTModel,
    MobileViTLayer,
    MobileViTForImageClassification,
    ViTModel,
    ViTLayer,
    ViTEmbeddings,
    ViTForImageClassification,
]:
    show_signature(cls)

show_source_snippet(MobileViTModel.forward, lines=28)
show_source_snippet(MobileViTLayer.forward, lines=26)
show_source_snippet(ViTEmbeddings.forward, lines=24)
show_source_snippet(ViTLayer.forward, lines=24)
show_source_snippet(ViTModel.forward, lines=28)


## Load pretrained checkpoints

This notebook uses:
- `apple/mobilevit-small` for MobileViT
- `google/vit-base-patch16-224` for a standard ViT baseline

Both models are loaded through `transformers` and run on the same demo image.


In [ ]:
mobilevit_ckpt = 'apple/mobilevit-small'
vit_ckpt = 'google/vit-base-patch16-224'

mobilevit_processor = AutoImageProcessor.from_pretrained(mobilevit_ckpt)
vit_processor = AutoImageProcessor.from_pretrained(vit_ckpt)

mobilevit_model = MobileViTForImageClassification.from_pretrained(mobilevit_ckpt).to(device).eval()
vit_model = ViTForImageClassification.from_pretrained(vit_ckpt).to(device).eval()

mobilevit_inputs = mobilevit_processor(images=demo_image, return_tensors='pt')
vit_inputs = vit_processor(images=demo_image, return_tensors='pt')

print_shape('MobileViT pixel_values', mobilevit_inputs['pixel_values'])
print_shape('ViT pixel_values', vit_inputs['pixel_values'])
print('MobileViT num_labels:', mobilevit_model.config.num_labels)
print('ViT num_labels:', vit_model.config.num_labels)


## Tensor walkthrough: image to logits

The walkthrough below prints the shape after every major transformation in MobileViT, starting from the preprocessed image tensor and ending at classification logits.

The helper functions explicitly step through the MobileNet-style stages and the MobileViT blocks so the tensor flow is easy to follow.


In [ ]:
def walk_generic_stage(stage, x, label):
    for child_name, child in stage.named_children():
        if isinstance(child, torch.nn.ModuleList):
            for index, block in enumerate(child):
                x = block(x)
                print_shape(f'{label}.{child_name}[{index}]', x)
        else:
            x = child(x)
            print_shape(f'{label}.{child_name}', x)
    return x

def walk_mobilevit_layer(layer, x, label):
    print(f'\n{label}')
    if getattr(layer, 'downsampling_layer', None) is not None:
        x = layer.downsampling_layer(x)
        print_shape(f'{label}.downsampling_layer', x)

    residual = x
    x = layer.conv_kxk(x)
    print_shape(f'{label}.conv_kxk', x)

    x = layer.conv_1x1(x)
    print_shape(f'{label}.conv_1x1', x)

    patches, info = layer.unfolding(x)
    print_shape(f'{label}.unfolding.patches', patches)
    patch_h = getattr(layer, 'patch_height', 'n/a')
    patch_w = getattr(layer, 'patch_width', 'n/a')
    print(
        f"{label}.unfolding.info{' ' * 10}patch={patch_h}x{patch_w}, "
        f"grid={info['num_patches_height']}x{info['num_patches_width']}, "
        f"orig_size={info['orig_size']}, interpolate={info['interpolate']}"
    )

    patches = layer.transformer(patches)
    print_shape(f'{label}.transformer', patches)

    patches = layer.layernorm(patches)
    print_shape(f'{label}.layernorm', patches)

    x = layer.folding(patches, info)
    print_shape(f'{label}.folding', x)

    x = layer.conv_projection(x)
    print_shape(f'{label}.conv_projection', x)

    x = layer.fusion(torch.cat((residual, x), dim=1))
    print_shape(f'{label}.fusion', x)
    return x

pixel_values = mobilevit_inputs['pixel_values'].to(device)
print_shape('input.image_processor_output', pixel_values)

with torch.inference_mode():
    x = mobilevit_model.mobilevit.conv_stem(pixel_values)
    print_shape('mobilevit.conv_stem', x)

    x = walk_generic_stage(mobilevit_model.mobilevit.encoder.layer[0], x, 'encoder.layer[0]')
    x = walk_generic_stage(mobilevit_model.mobilevit.encoder.layer[1], x, 'encoder.layer[1]')

    x = walk_mobilevit_layer(mobilevit_model.mobilevit.encoder.layer[2], x, 'encoder.layer[2]')
    x = walk_mobilevit_layer(mobilevit_model.mobilevit.encoder.layer[3], x, 'encoder.layer[3]')
    x = walk_mobilevit_layer(mobilevit_model.mobilevit.encoder.layer[4], x, 'encoder.layer[4]')

    x = mobilevit_model.mobilevit.conv_1x1_exp(x)
    print_shape('mobilevit.conv_1x1_exp', x)

    pooled = torch.mean(x, dim=(-2, -1))
    print_shape('mobilevit.global_average_pool', pooled)

    dropped = mobilevit_model.dropout(pooled)
    print_shape('mobilevit.dropout', dropped)

    logits = mobilevit_model.classifier(dropped)
    print_shape('mobilevit.logits', logits)

    top5 = torch.topk(logits.softmax(dim=-1), k=5, dim=-1)
    print('Top-5 class indices:', top5.indices[0].tolist())
    print('Top-5 probabilities:', [round(v, 4) for v in top5.values[0].tolist()])


## Runtime comparison: MobileViT vs. ViT

The benchmark below times the backbone forward pass only.

That keeps the comparison focused on the encoder itself rather than the tiny classification head.
Both models still start from the same demo image.


In [ ]:
def benchmark_encoder(model, pixel_values, runs: int = 10, warmup: int = 3):
    model.eval()
    pixel_values = pixel_values.to(device)

    with torch.inference_mode():
        for _ in range(warmup):
            _ = model(pixel_values=pixel_values)

        if device.type == 'cuda':
            torch.cuda.synchronize()

        start = time.perf_counter()
        for _ in range(runs):
            _ = model(pixel_values=pixel_values)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        end = time.perf_counter()

    return (end - start) / runs

mobilevit_backbone = mobilevit_model.mobilevit
vit_backbone = vit_model.vit

mobilevit_seconds = benchmark_encoder(mobilevit_backbone, mobilevit_inputs['pixel_values'])
vit_seconds = benchmark_encoder(vit_backbone, vit_inputs['pixel_values'])
delta_seconds = vit_seconds - mobilevit_seconds

print(f'MobileViT backbone average runtime: {mobilevit_seconds:.4f} s/run')
print(f'ViT backbone average runtime:       {vit_seconds:.4f} s/run')
print(f'Difference (ViT - MobileViT):       {delta_seconds:+.4f} s/run')

if delta_seconds > 0:
    print(f'MobileViT is faster by about {delta_seconds:.4f} seconds per run on this machine.')
else:
    print(f'ViT is faster by about {abs(delta_seconds):.4f} seconds per run on this machine.')
